# TalkingData Ad-Click Fraud: Phase 1 EDA

This notebook examines the reproducible, **contiguous and time-ordered** 1M-click window at `../data/clicks_sample.csv`. TalkingData's `is_attributed` is an install/conversion outcome, **not** the fraud target. `is_fraud` is a clearly marked bootstrap heuristic for this portfolio project.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/clicks_sample.csv')
METADATA = Path('../data/clicks_sample_metadata.json')
df = pd.read_csv(DATA, parse_dates=['click_time', 'timestamp', 'attributed_time'])
metadata = json.loads(METADATA.read_text())
df.shape, metadata

## Label construction

A row receives `is_fraud = 1` only when its IP is in the window's top 0.1% velocity tail **and** has an attribution rate at or below 0.1%. All other rows are provisionally negative. This identifies a click-farm-like signature; it is not a claim that every unattributed click is fraudulent. Synthetic farms, when added in Phase 4, are separately marked with `is_synthetic`.

In [ ]:
summary = pd.Series({
    'clicks': len(df),
    'bootstrap fraud rate': df.is_fraud.mean(),
    'attribution rate': df.is_attributed.mean(),
    'unique IPs': df.ip.nunique(),
    'window start': df.click_time.min(),
    'window end': df.click_time.max(),
})
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Class imbalance
df.is_fraud.value_counts().rename({0: 'bootstrap negative', 1: 'bootstrap positive'}).plot.bar(ax=axes[0, 0], title='Bootstrap fraud-label balance')

# Click-time distribution
df.set_index('click_time').resample('1min').size().plot(ax=axes[0, 1], title='Click volume by minute')
axes[0, 1].set_ylabel('clicks')

# Campaign (channel) is also the documented publisher proxy in v1.
campaign = df.groupby('campaign_id').agg(clicks=('click_id', 'size'), fraud_rate=('is_fraud', 'mean'))
campaign.query('clicks >= 100').nlargest(15, 'fraud_rate').fraud_rate.sort_values().plot.barh(ax=axes[1, 0], title='Highest fraud-rate campaigns (>=100 clicks)')

# Farm signature: high click volume with low conversion.
ip = df.groupby('ip').agg(clicks=('click_id', 'size'), attribution_rate=('is_attributed', 'mean'), fraud_rate=('is_fraud', 'mean'))
axes[1, 1].scatter(ip.clicks, ip.attribution_rate, c=ip.fraud_rate, s=8, alpha=.35, cmap='Reds')
axes[1, 1].set_xscale('log')
axes[1, 1].set_xlabel('click volume per IP (log)')
axes[1, 1].set_ylabel('attribution rate per IP')
axes[1, 1].set_title('Conversion rate versus IP click volume')

plt.tight_layout()

## Scope caveats

`channel` is used as both campaign and publisher proxy because TalkingData does not expose independent publisher metadata. The source does not provide user agent, referrer, IP geolocation, or device locale; those remain out of the v1 model unless explicitly added on synthetic farm events.